# Notebook 02 — Build the Label

## Objective

Create the target label for delivery prediction.

## Label

- `late = 1` → delivered after the estimated date
- `late = 0` → delivered on or before the estimated date
- `late = NaN` → required delivery information is unavailable

## Input

`artifacts/notebook_01/ml_table.parquet`

## Output

`artifacts/notebook_02/ml_table_labeled.parquet`

## Grain

One row = one order

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
input_path = Path("../artifacts/notebook_01/ml_table.parquet")

ml_table = pd.read_parquet(input_path)

print("Loaded ML table successfully.")
print("Shape:", ml_table.shape)

Loaded ML table successfully.
Shape: (99441, 19)


In [3]:
required_columns = [
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

missing_columns = [
    col for col in required_columns
    if col not in ml_table.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [4]:
print(
    ml_table[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ].dtypes
)

order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    ml_table[col] = pd.to_datetime(
        ml_table[col],
        errors="coerce"
    )

In [6]:
print(ml_table[date_columns].dtypes)

order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [7]:
print("Missing actual delivery dates:")
print(
    ml_table["order_delivered_customer_date"].isna().sum()
)

print("\nMissing estimated delivery dates:")
print(
    ml_table["order_estimated_delivery_date"].isna().sum()
)

Missing actual delivery dates:
2965

Missing estimated delivery dates:
0


In [8]:
missing_actual_delivery = ml_table[
    ml_table["order_delivered_customer_date"].isna()
]

print(
    missing_actual_delivery["order_status"]
    .value_counts(dropna=False)
)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [9]:
ml_table["label_data_available"] = (
    ml_table["order_delivered_customer_date"].notna()
    & ml_table["order_estimated_delivery_date"].notna()
)
print(
    ml_table["label_data_available"]
    .value_counts()
)

label_data_available
True     96476
False     2965
Name: count, dtype: int64


In [10]:
ml_table["delivery_delay_days"] = (
    ml_table["order_delivered_customer_date"]
    - ml_table["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

ml_table["delivery_delay_days"].describe()


count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64

In [11]:
ml_table["late"] = np.nan

valid_label = ml_table["label_data_available"]

ml_table.loc[valid_label, "late"] = (
    ml_table.loc[valid_label, "delivery_delay_days"] > 0
).astype(int)

In [12]:
print(
    ml_table["late"]
    .value_counts(dropna=False)
    .sort_index()
)

late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64


In [13]:
label_distribution = (
    ml_table["late"]
    .value_counts(normalize=True, dropna=False)
    * 100
)

print(label_distribution)

late
0.0    89.147334
1.0     7.870999
NaN     2.981668
Name: proportion, dtype: float64


In [14]:
print("Late orders:")
print((ml_table["late"] == 1).sum())

print("\nOn-time orders:")
print((ml_table["late"] == 0).sum())

print("\nOrders without label:")
print(ml_table["late"].isna().sum())

Late orders:
7827

On-time orders:
88649

Orders without label:
2965


In [15]:
label_summary = pd.DataFrame({
    "label": [0, 1],
    "meaning": ["On Time", "Late"],
    "count": [
        (ml_table["late"] == 0).sum(),
        (ml_table["late"] == 1).sum(),
    ],
})

label_summary["percentage_of_labeled"] = (
    label_summary["count"]
    / ml_table["late"].notna().sum()
    * 100
)

label_summary

,label,meaning,count,percentage_of_labeled
0,0,On Time,88649,91.887101
1,1,Late,7827,8.112899


In [16]:
assert (
    (ml_table["late"] == 1).sum()
    + (ml_table["late"] == 0).sum()
    + ml_table["late"].isna().sum()
) == len(ml_table)

print("Label accounting validation passed.")

Label accounting validation passed.


In [17]:
valid_labels = ml_table["late"].dropna().unique()

print("Valid label values:", sorted(valid_labels))

Valid label values: [np.float64(0.0), np.float64(1.0)]


In [18]:
assert set(valid_labels).issubset({0.0, 1.0})

print("Label values validation passed.")

Label values validation passed.


In [19]:
late_check = ml_table.loc[
    ml_table["late"].notna(),
    ["delivery_delay_days", "late"]
]

print(
    late_check.groupby("late")["delivery_delay_days"]
    .agg(["min", "max"])
)

             min         max
late                        
0.0  -146.016123   -0.000058
1.0     0.002500  188.975081


In [20]:
assert (
    ml_table.loc[ml_table["late"] == 1, "delivery_delay_days"] > 0
).all()

assert (
    ml_table.loc[ml_table["late"] == 0, "delivery_delay_days"] <= 0
).all()

print("Delay-label consistency validation passed.")

Delay-label consistency validation passed.


In [21]:
print("Rows:", len(ml_table))
print("Unique orders:", ml_table["order_id"].nunique())
print("Order IDs unique:", ml_table["order_id"].is_unique)

Rows: 99441
Unique orders: 99441
Order IDs unique: True


In [22]:
assert len(ml_table) == ml_table["order_id"].nunique()
assert ml_table["order_id"].is_unique

print("Grain validation passed: one row per order.")

Grain validation passed: one row per order.


In [23]:
artifact_dir = Path("../artifacts/notebook_02")
artifact_dir.mkdir(parents=True, exist_ok=True)

In [24]:
output_path = artifact_dir / "ml_table_labeled.parquet"

ml_table.to_parquet(
    output_path,
    index=False
)

print(f"Saved labeled ML table to: {output_path}")

Saved labeled ML table to: ..\artifacts\notebook_02\ml_table_labeled.parquet


In [25]:
saved_ml_table = pd.read_parquet(output_path)

print("Saved table shape:", saved_ml_table.shape)
print("Unique orders:", saved_ml_table["order_id"].nunique())
print("Order IDs unique:", saved_ml_table["order_id"].is_unique)

print("\nLabel values:")
print(saved_ml_table["late"].value_counts(dropna=False))

Saved table shape: (99441, 22)
Unique orders: 99441
Order IDs unique: True

Label values:
late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64


## Summary

- Loaded the order-level ML table.
- Converted delivery dates to datetime.
- Calculated delivery delay.
- Created the `late` target.
- Preserved orders with unavailable labels.
- Validated the target and one-row-per-order grain.

### Output

`artifacts/notebook_02/ml_table_labeled.parquet`